# CAWOT-CM V0 — End-to-end Colab demo

Pipeline:
1. Extract CLIP embeddings of the training pool
2. FAISS k-means clustering
3. Two coresets: **Random** (baseline) vs **V0** (farthest-from-centroid)
4. Fine-tune CLIP on each coreset
5. Evaluate text→image retrieval (R@1/5/10, mAP)

**Two run modes** (pick one in step 4):
- **Sanity mode** — generate dummy data, verify pipeline runs end-to-end (~10 min on T4)
- **Real mode** — use PAB dataset from Google Drive

Recommended: run sanity mode first to make sure everything works, then switch to real PAB.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Upload code

Two options:

**Option A (recommended): upload the `cawot-cm-v0` folder via the Files panel** (left sidebar → upload folder), then run the next cell.

**Option B: zip + upload single file**, then unzip:
```
!unzip -q cawot-cm-v0.zip -d /content/
```

In [ ]:
import os
PROJECT_DIR = "/content/cawot-cm-v0"
assert os.path.exists(PROJECT_DIR), f"Upload cawot-cm-v0 to {PROJECT_DIR} first"
os.chdir(PROJECT_DIR)
!ls

## 3. Install dependencies

Colab already has torch/torchvision. We add open_clip, faiss-gpu, etc. ~2 minutes.

In [ ]:
!pip install -q open_clip_torch faiss-gpu-cu12 einops pyyaml wandb tqdm

## 4. Choose run mode

Set `MODE` to `"sanity"` or `"pab"`.

In [ ]:
MODE = "sanity"   # "sanity" | "pab"

if MODE == "pab":
    # PAB on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/pab_data"   # adjust to your Drive path
    SUBSET_SIZE = 50_000   # set to None for full 1M
    K_CLUSTERS = 500       # 5000 for full 1M
elif MODE == "sanity":
    DATA_ROOT = "/content/cawot-cm-v0/dummy_pab"
    SUBSET_SIZE = None
    K_CLUSTERS = 50
    !python scripts/make_dummy_data.py --root {DATA_ROOT} --n-train 2000 --n-query 100 --n-gallery 500
else:
    raise ValueError(MODE)

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"SUBSET_SIZE = {SUBSET_SIZE}")
print(f"K_CLUSTERS = {K_CLUSTERS}")

## 5. Patch config.yaml with the chosen settings

In [ ]:
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"] = DATA_ROOT
cfg["data"]["subset_size"] = SUBSET_SIZE
cfg["cluster"]["k"] = K_CLUSTERS

if MODE == "sanity":
    # Speed knobs for the dummy run
    cfg["train"]["num_epochs"] = 1
    cfg["train"]["batch_size"] = 64
    cfg["embed"]["batch_size"] = 128
    cfg["eval"]["batch_size"] = 128
    cfg["data"]["num_workers"] = 2
    cfg["coreset"]["budget_ratio"] = 0.2

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(yaml.dump(cfg, sort_keys=False))

## 6. Run end-to-end

Sanity mode: ~5-10 min on T4.
Real PAB at 50K subset: ~45-90 min.

Steps are cached — safe to re-run.

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 7. Inspect results

In [ ]:
import json, pandas as pd
with open("outputs/eval/summary.json") as f:
    summary = json.load(f)
df = pd.DataFrame(summary).T
df

## 8. Interpreting results

**Sanity mode**: the absolute numbers don't mean anything — data is random. What matters is that the pipeline runs end-to-end with no errors and produces a `summary.json`. If both Random and V0 give roughly similar metrics here, that's fine — there's no real signal in dummy data.

**Real PAB mode**: expectations at 20% budget on the full 1M pool:

| Method | Expected R@1 |
|--------|-------------|
| Random | ~78-79% |
| V0     | ~79-80% (+0.5-1.5%) |

**Caveats with this V0 codebase**:
- V0 uses **CLIP + last-4-layer finetune + InfoNCE**, not IRRA. Absolute numbers will be lower than the CMP paper (which uses full IRRA finetune). What matters is the **relative** ranking V0 > Random.
- At 50K subset (Colab demo), expect lower absolute numbers but the V0 > Random gap should still appear.
- If V0 ≤ Random on real PAB → there's a bug in selection logic OR k_clusters is wrong scale. Sanity check before V1.

## 9. Next steps

1. **Tune hyperparameters** on the 50K subset (epochs, lr, K).
2. **Run at full 1M** on ThunderCompute when results are convincing.
3. **Add more baselines**: SemDeDup, k-center, CLIPScore filter.
4. **Start V1**: cross-modal cost + submodlib facility-location.